In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [6]:


# ==========================================
# 1. 準備資料 (包含 Lag 製作)
# ==========================================
print("Loading data and creating features...")
df = pd.read_csv('numeric_log_transformed.csv', index_col=0, parse_dates=True)

cols_to_drop = [
    'SP500 30 Day Volatility',
    'SPX Put Volume',
    'Total SPX Options Volume',
    'VIX'
]
# 檢查欄位是否存在再刪除，避免報錯
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

target_col = 'SP500 Log Returns'

# --- 【補回這段】製作 Lag 特徵 ---
max_lag = 7  # 根據你之前的設定
df_original = df.copy()

for lag in range(1, max_lag + 1):
    lagged = df_original.shift(lag)
    lagged.columns = [f'{col}_lag{lag}' for col in df_original.columns]
    df = pd.concat([df, lagged], axis=1)

# 移除因為 Shift 產生的 NaN (前7筆)
df = df.dropna()

# 現在 df 裡面才有 _lag 欄位，這行才會有東西
feature_cols = [c for c in df.columns if '_lag' in c]

X = df[feature_cols]
y = df[target_col]

# 檢查一下確保有東西 (Debug 用)
print(f"Features created: {len(feature_cols)}")
print(f"Data shape: {X.shape}") 
# 如果這裡印出 Features created: 0，那就是欄位名稱不對，要檢查一下 csv

# ==========================================
# 2. 設定 Time Series Split
# ==========================================
tscv = TimeSeriesSplit(n_splits=5)
fold_metrics = [] 
trading_results_list = [] 

print("Starting Benchmark (Linear Regression with Time Series Split)...")

# ==========================================
# 3. 執行交叉驗證
# ==========================================
for fold_idx, (train_index, test_index) in enumerate(tscv.split(X)):
    
    # A. 切分
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # B. 標準化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # C. 訓練
    model = LinearRegression() 
    model.fit(X_train_scaled, y_train)
    
    # D. 預測
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    
    # E. 計算指標
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_da = np.mean(np.sign(y_test_pred) == np.sign(y_test))
    
    # F. 存入統計指標
    fold_metrics.append({
        'Fold': fold_idx + 1,
        'Test MAE': test_mae,
        'Test RMSE': test_rmse,
        'Test DA': test_da
    })
    
    # G. 存入詳細預測數據 (為了做回測)
    fold_df = pd.DataFrame({
        'Actual': y_test,
        'Predicted': y_test_pred
    }, index=y_test.index)
    
    trading_results_list.append(fold_df)
    
    print(f"Fold {fold_idx+1} | Test MAE: {test_mae:.6f} | Test DA: {test_da:.4f}")

# ==========================================
# 4. 計算最終 Benchmark 平均分
# ==========================================
df_bm_folds = pd.DataFrame(fold_metrics)
avg_benchmark = df_bm_folds.mean()

print("\n" + "="*40)
print("FINAL BENCHMARK SCORE")
print(f"Average Test MAE:  {avg_benchmark['Test MAE']:.6f}")
print(f"Average Test RMSE:  {avg_benchmark['Test RMSE']:.6f}")
print(f"Average Test DA:   {avg_benchmark['Test DA']:.4f}")


Loading data and creating features...
Features created: 210
Data shape: (2257, 210)
Starting Benchmark (Linear Regression with Time Series Split)...
Fold 1 | Test MAE: 0.049941 | Test DA: 0.4574
Fold 2 | Test MAE: 0.010016 | Test DA: 0.5080
Fold 3 | Test MAE: 0.008802 | Test DA: 0.5106
Fold 4 | Test MAE: 0.017769 | Test DA: 0.4362
Fold 5 | Test MAE: 0.007613 | Test DA: 0.5266

FINAL BENCHMARK SCORE
Average Test MAE:  0.018828
Average Test RMSE:  0.022619
Average Test DA:   0.4878
